# Problem Statement:


$$A \mathbf{x} = b$$

Mentality:

Triangular elimination of a 3x3 matrix A with rotational matrix of G1 G2 G3

heres a simple visualisation (forward pass, need to find deg 1,2,3)

In [ ]:
import numpy as np

def get_cos_sin(degree: int = 90):
    theta = np.radians(degree)  # Convert degrees to radians
    c, s = np.cos(theta), np.sin(theta)
    return c, s

deg1 = 90
deg2 = 45
deg3 = 60

cos1, sin1 = get_cos_sin(deg1)
cos2, sin2 = get_cos_sin(deg2)
cos3, sin3 = get_cos_sin(deg3)

A = np.array([
    [3, 4, 5],
    [-1, 1, 3],
    [2, -2, 3],
])

b = np.array([7, 1, 5,])

G1 = np.array([
    [cos1, 0, -sin1],
    [0, 1, 0],
    [sin1, 0, cos1],
])

G2 = np.array([
    [cos2, -sin2, 0],
    [sin2, cos2, 0],
    [0, 0, 1],
])

G3 = np.array([
    [cos3, 0, -sin3],
    [0, 1, 0],
    [sin3, 0, cos3],
])

G1 @ A

array([[-2.,  2., -3.],
       [-1.,  1.,  3.],
       [ 3.,  4.,  5.]])

To zero out entries and transform $A$ into an upper-triangular matrix using rotation matrices, you are implementing **Givens Rotations** (QR decomposition via rotations).

Hardcoded angles like $90^\circ, 45^\circ, 60^\circ$ will not work. The values $c = \cos\theta$ and $s = \sin\theta$ must be calculated directly from the matrix entries you want to eliminate.

---

# Mechanics of a Givens Rotation

To zero out an element $b$ using an element $a$ above it, we want:

$$\begin{bmatrix} c & s \\ -s & c \end{bmatrix} \begin{bmatrix} a \\ b \end{bmatrix} = \begin{bmatrix} r \\ 0 \end{bmatrix}$$

Solving for $c$ and $s$:


$$r = \sqrt{a^2 + b^2}, \quad c = \frac{a}{r}, \quad s = \frac{b}{r}$$

Notice that:


$$-s \cdot a + c \cdot b = -\left(\frac{b}{r}\right)a + \left(\frac{a}{r}\right)b = 0$$

You never need to compute $\theta$ or call `math.sin` / `math.cos`.

---

### Step-by-Step Triangle Elimination for $3 \times 3$

The subdiagonal entries to eliminate in order are:

1. Position $(1, 0)$ (row 1, col 0) $\rightarrow -1$
2. Position $(2, 0)$ (row 2, col 0) $\rightarrow 2$
3. Position $(2, 1)$ (row 2, col 1) $\rightarrow$ current value after steps 1 & 2

#### Step 1: Zero out $A[1, 0] = -1$ using $A[0, 0] = 3$

$$a = 3, \quad b = -1$$

$$r = \sqrt{3^2 + (-1)^2} = \sqrt{10}$$

$$c = \frac{3}{\sqrt{10}}, \quad s = \frac{-1}{\sqrt{10}}$$

The rotation matrix acting on rows 0 and 1 is:

$$G_{10} = \begin{bmatrix} c & s & 0 \\ -s & c & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

Apply transformation:


$$A^{(1)} = G_{10} A, \quad b^{(1)} = G_{10} b$$

---

#### Step 2: Zero out $A^{(1)}[2, 0]$ using $A^{(1)}[0, 0]$

Using updated elements $a = A^{(1)}[0, 0]$ and $b = A^{(1)}[2, 0]$:


$$r = \sqrt{a^2 + b^2}, \quad c = \frac{a}{r}, \quad s = \frac{b}{r}$$

The rotation matrix acting on rows 0 and 2 is:

$$G_{20} = \begin{bmatrix} c & 0 & s \\ 0 & 1 & 0 \\ -s & 0 & c \end{bmatrix}$$

Apply transformation:


$$A^{(2)} = G_{20} A^{(1)}, \quad b^{(2)} = G_{20} b^{(1)}$$

Now the first column below the diagonal is all zeros.

---

#### Step 3: Zero out $A^{(2)}[2, 1]$ using $A^{(2)}[1, 1]$

Using updated elements $a = A^{(2)}[1, 1]$ and $b = A^{(2)}[2, 1]$:


$$r = \sqrt{a^2 + b^2}, \quad c = \frac{a}{r}, \quad s = \frac{b}{r}$$

The rotation matrix acting on rows 1 and 2 is:

$$G_{21} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & c & s \\ 0 & -s & c \end{bmatrix}$$

Apply transformation:


$$R = G_{21} A^{(2)}, \quad b_{\text{new}} = G_{21} b^{(2)}$$

$R$ is now strictly upper-triangular.

In [35]:
import numpy as np

def givens_coefficients(a, b):
    r = np.hypot(a, b)
    if r == 0:
        return 1.0, 0.0
    return a / r, b / r

def make_givens(n, i, j, c, s):
    """Creates an n x n Givens rotation matrix affecting rows i and j."""
    G = np.eye(n)
    G[i, i] = c
    G[i, j] = s
    G[j, i] = -s
    G[j, j] = c
    return G

A = np.array([
    [3.0,  4.0, 5.0],
    [-1.0, 1.0, 3.0],
    [2.0, -2.0, 3.0]
])
b = np.array([7.0, 1.0, 5.0])

A_orig = A.copy()
b_orig = b.copy()

# Sequence of eliminations: (target_row, pivot_row)
eliminations = [(1, 0), (2, 0), (2, 1)]

for step, (j, i) in enumerate(eliminations):
    c, s = givens_coefficients(A[i, i], A[j, i])
    G = make_givens(len(A), i, j, c, s)
    A = G @ A
    b = G @ b

    print(f"\nStep {step} -> Row elimination: row: {j} col: {i}")
    print("Upper-triangular matrix R:")
    print(np.round(A, 4))

    print("Transformed vector b:")
    print(np.round(b, 4))


Step 0 -> Row elimination: row: 1 col: 0
Upper-triangular matrix R:
[[ 3.1623  3.4785  3.7947]
 [ 0.      2.2136  4.4272]
 [ 2.     -2.      3.    ]]
Transformed vector b:
[6.3246 3.1623 5.    ]

Step 1 -> Row elimination: row: 2 col: 0
Upper-triangular matrix R:
[[ 3.7417  1.8708  4.8107]
 [ 0.      2.2136  4.4272]
 [ 0.     -3.5496  0.5071]]
Transformed vector b:
[8.0178 3.1623 0.8452]

Step 2 -> Row elimination: row: 2 col: 1
Upper-triangular matrix R:
[[3.7417 1.8708 4.8107]
 [0.     4.1833 1.9124]
 [0.     0.     4.0249]]
Transformed vector b:
[8.0178 0.9562 3.1305]


Once in upper-triangular form $R x = b_{\text{new}}$, you can solve for $x$ directly via back-substitution.

# Relating Implementation

Once $A$ is transformed into an upper-triangular matrix $R$ and $b$ is transformed into $\tilde{b}$, the original linear system:

$$A \mathbf{x} = b$$

becomes:

$$R \mathbf{x} = \tilde{b}$$

Because $Q = G_{21} G_{20} G_{10}$ is orthogonal ($Q^{-1} = Q^T$), multiplying both sides by $Q$ preserves the exact solution:

$$Q A \mathbf{x} = Q b \implies R \mathbf{x} = \tilde{b}$$

---

## Matrix Representation

In expanded form:

$$\begin{bmatrix} r_{00} & r_{01} & r_{02} \\ 0      & r_{11} & r_{12} \\ 0      & 0      & r_{22} \end{bmatrix} \begin{bmatrix} x \\ y \\ z \end{bmatrix} = \begin{bmatrix} \tilde{b}_0 \\ \tilde{b}_1 \\ \tilde{b}_2 \end{bmatrix}$$

This corresponds to three algebraic equations:

1. $r_{00} x + r_{01} y + r_{02} z = \tilde{b}_0$
2. $r_{11} y + r_{12} z = \tilde{b}_1$
3. $r_{22} z = \tilde{b}_2$

---

## Back-Substitution Algorithm

Solve from bottom to top:

1. **Solve for $z$ (row 2):**

$$z = \frac{\tilde{b}_2}{r_{22}}$$


2. **Solve for $y$ (row 1):**

$$y = \frac{\tilde{b}_1 - r_{12} z}{r_{11}}$$


3. **Solve for $x$ (row 0):**

$$x = \frac{\tilde{b}_0 - r_{01} y - r_{02} z}{r_{00}}$$

In [36]:

import numpy as np

def back_substitution(R, b):
    """
    Solves R x = b for an upper-triangular matrix R of shape (n, n)
    and vector b of shape (n,).
    """
    n = len(b)
    x = np.zeros(n, dtype=np.float64)

    # Traverse backwards from n-1 down to 0
    for i in range(n - 1, -1, -1):
        if np.isclose(R[i, i], 0.0):
            raise ValueError(f"Matrix is singular: zero pivot at row {i}")
        
        # Subtract already known variables
        known_sum = np.dot(R[i, i + 1:], x[i + 1:])
        x[i] = (b[i] - known_sum) / R[i, i]

    return x


# Continuing from the Givens rotation output:
# R is the upper-triangular matrix, b is the transformed RHS vector
x_sol = back_substitution(A, b)

print("Solution [x, y, z]:")
print(x_sol)


Solution [x, y, z]:
[ 1.20634921 -0.12698413  0.77777778]


## Verification

Check the residual against the original matrix and vector $A_{\text{orig}}$ and $b_{\text{orig}}$:

In [43]:
print(f"A @ x = {A_orig @ x_sol}")
print(f"b = {b_orig}")

residual = A_orig @ x_sol - b_orig
print("Residual norm:", np.linalg.norm(residual))

A @ x = [7. 1. 5.]
b = [7. 1. 5.]
Residual norm: 1.1102230246251565e-15
